# Decoder Comparison — U-Net vs V-Net (cross-validated, with significance)

This notebook **aggregates** the per-knee metric CSVs written by every `03_decoder_pipeline.ipynb` run
and produces the headline U-Net vs V-Net comparison. It does **no training** — run it after the grid
of decoder runs is finished.

**Inputs it reads:** `models/decoders/fold{K}/{regime}/{model}/{model}_test_metrics.csv`, one row per
held-out test knee, with columns `model, fold, regime, dataset, case, side, dice, iou, hd95_mm,
assd_mm`. Because the comparison uses **k-fold CV**, pooling these CSVs gives **one score per knee**
(every knee is a test knee in exactly one fold), so the fractured subgroup is evaluated on *all*
fractured knees rather than the ~2 a single split would leave.

**What it produces:**
1. **mean ± std** per `regime` × group (overall / healthy / fractured) × model — the deliverable table.
2. A **paired Wilcoxon signed-rank test** (U-Net vs V-Net, paired by knee) per metric, with
   **Holm-Bonferroni** correction across the metric family — overall and within the fractured subgroup.

**Why these stats.** The Wilcoxon signed-rank test is the standard non-parametric paired test for
comparing two segmentation models on identical test cases without assuming normal metric
distributions (e.g. the U-Net benchmarking literature; Holm-Bonferroni controls the family-wise error
across the 4 metrics). With only ~13 fractured knees a single split is uninformative, so CV + a paired
test is the minimum needed for a defensible claim (FracReconNet, PMC9829664; RSNA Radiology:AI 2022).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 140); pd.set_option("display.max_columns", 30)

# --- locate models/decoders and pool every per-knee metric CSV ---
def find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "models" / "decoders").exists():
            return cand
    raise FileNotFoundError("Could not find models/decoders. Run the decoder grid first.")

ROOT = find_root(Path.cwd())
DEC = ROOT / "models" / "decoders"
csvs = sorted(DEC.glob("fold*/*/*/*_test_metrics.csv"))   # fold{K}/{regime}/{model}/{model}_test_metrics.csv
print("found %d per-knee metric CSV(s) under %s" % (len(csvs), DEC))

METRICS = ["dice", "iou", "hd95_mm", "assd_mm"]   # higher Dice/IoU = better; lower HD95/ASSD = better
LOWER_IS_BETTER = {"dice": False, "iou": False, "hd95_mm": True, "assd_mm": True}

if csvs:
    df = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
    # one score per (regime, model, knee); a knee is a test knee in exactly one fold
    df = df.drop_duplicates(subset=["regime", "model", "dataset", "case", "side"])
    print("pooled rows:", len(df))
    print("regimes:", sorted(df.regime.unique()), "| models:", sorted(df.model.unique()))
    print("knees tested per regime/model/group:\n",
          df.groupby(["regime", "model", "dataset"]).size())
else:
    df = pd.DataFrame(columns=["model", "fold", "regime", "dataset", "case", "side"] + METRICS)
    print("[warn] no metric CSVs yet — run 03_decoder_pipeline.ipynb across folds/regimes/models first.")

found 0 per-knee metric CSV(s) under C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\models\decoders
[warn] no metric CSVs yet — run 03_decoder_pipeline.ipynb across folds/regimes/models first.


## 1. Headline table — mean ± std by regime, group, model

For each front-end **regime** (`frozen` = strict decoder isolation; `finetuned` = realistic
capacity), we report each metric's **mean ± std across the cross-validated knees**, split by group:
`all`, `healthy`, `fractured`. The fractured row is the one the research question turns on.

In [2]:
# Add an "all" group alongside healthy/fractured, then format mean +/- std per metric.
def mean_std_table(d):
    if len(d) == 0:
        return pd.DataFrame()
    d_all = d.copy(); d_all["dataset"] = "all"
    dd = pd.concat([d, d_all], ignore_index=True)
    g = dd.groupby(["regime", "dataset", "model"])
    out = {}
    for m in METRICS:
        mean = g[m].mean(); std = g[m].std()
        out[m] = mean.round(3).astype(str) + " +/- " + std.round(3).fillna(0).astype(str)
    tbl = pd.DataFrame(out)
    tbl["n_knees"] = g.size()
    # order rows: regime, then all/healthy/fractured, then model
    order = {"all": 0, "healthy": 1, "fractured": 2}
    tbl = tbl.reset_index()
    tbl["__o"] = tbl["dataset"].map(order)
    tbl = tbl.sort_values(["regime", "__o", "model"]).drop(columns="__o").set_index(
        ["regime", "dataset", "model"])
    return tbl

table = mean_std_table(df)
if len(table):
    print("U-Net vs V-Net — mean +/- std over cross-validated knees\n")
    print(table.to_string())
    table.to_csv(DEC / "comparison_summary.csv")
    print("\nwrote ->", DEC / "comparison_summary.csv")
else:
    print("no data to aggregate yet.")

no data to aggregate yet.


## 2. Significance — paired Wilcoxon signed-rank (U-Net vs V-Net)

We pair U-Net and V-Net by the **same knee** (each knee appears once per model across the CV folds)
and run a two-sided **Wilcoxon signed-rank test** per metric, separately for each regime and for the
`all` / `healthy` / `fractured` groups. P-values are corrected within each test family with
**Holm-Bonferroni**. `p_holm < 0.05` means the two decoders differ significantly on that metric;
otherwise the comparison cannot distinguish them (the likely outcome on the small fractured subgroup —
which is itself the finding: residual-block choice does not rescue fracture reconstruction here).

In [3]:
from scipy.stats import wilcoxon

def holm_bonferroni(pvals):
    """Holm step-down adjusted p-values (NaNs passed through)."""
    p = np.asarray(pvals, dtype=float)
    ok = ~np.isnan(p); adj = np.full_like(p, np.nan)
    idx = np.where(ok)[0]; order = idx[np.argsort(p[idx])]
    m = len(order); run = 0.0
    for rank, j in enumerate(order):
        run = max(run, (m - rank) * p[j]); adj[j] = min(run, 1.0)
    return adj

def paired_vectors(d, metric):
    """Returns aligned U-Net / V-Net metric vectors paired by knee (regime fixed upstream)."""
    piv = d.pivot_table(index=["dataset", "case", "side"], columns="model", values=metric)
    if not {"unet", "vnet"}.issubset(piv.columns):
        return None, None
    piv = piv.dropna(subset=["unet", "vnet"])
    return piv["unet"].to_numpy(), piv["vnet"].to_numpy()

rows = []
if len(df):
    for regime in sorted(df.regime.unique()):
        dr = df[df.regime == regime]
        for group in ["all", "healthy", "fractured"]:
            dg = dr if group == "all" else dr[dr.dataset == group]
            praw = []
            for m in METRICS:
                u, v = paired_vectors(dg, m)
                rec = dict(regime=regime, group=group, metric=m,
                           n_pairs=(0 if u is None else len(u)),
                           unet_mean=(np.nan if u is None else round(float(np.mean(u)), 3)),
                           vnet_mean=(np.nan if v is None else round(float(np.mean(v)), 3)),
                           stat=np.nan, p_raw=np.nan)
                if u is not None and len(u) >= 1 and np.any(u - v != 0):
                    try:
                        s, p = wilcoxon(u, v)            # two-sided, drops zero-differences
                        rec["stat"] = float(s); rec["p_raw"] = float(p)
                    except ValueError:
                        pass                              # all-zero diffs / too few pairs
                praw.append(rec["p_raw"]); rows.append(rec)
            # Holm-correct within this regime x group family (the 4 metrics)
            adj = holm_bonferroni(praw)
            for k in range(4):
                rows[-4 + k]["p_holm"] = (None if np.isnan(adj[k]) else round(float(adj[k]), 4))

    res = pd.DataFrame(rows)
    def better(r):
        if pd.isna(r.p_raw):
            return "n/a"
        win = "vnet" if ((r.vnet_mean > r.unet_mean) ^ LOWER_IS_BETTER[r.metric]) else "unet"
        sig = (r.get("p_holm") is not None) and (r["p_holm"] < 0.05)
        return ("%s*" % win) if sig else ("%s (ns)" % win)
    res["better"] = res.apply(better, axis=1)
    res["p_raw"] = res["p_raw"].round(4)
    print("Paired Wilcoxon (U-Net vs V-Net), Holm-corrected per regime x group:\n")
    print(res[["regime", "group", "metric", "n_pairs", "unet_mean", "vnet_mean",
               "p_raw", "p_holm", "better"]].to_string(index=False))
    res.to_csv(DEC / "comparison_wilcoxon.csv", index=False)
    print("\n'*' = significant at Holm p<0.05; '(ns)' = direction only, not significant.")
    print("wrote ->", DEC / "comparison_wilcoxon.csv")
else:
    print("no data — run the decoder grid first.")

no data — run the decoder grid first.


## 3. Per-bone breakdown (femur / tibia / patella / fibula)

The per-knee CSVs now carry per-bone columns (`dice_<bone>`, `iou_<bone>`, `hd95_mm_<bone>`,
`assd_mm_<bone>`) alongside the bone-averaged aggregates used in Sections 1-2. Here we report each
metric **per bone**, so the fracture-relevant bones can be inspected individually (e.g. patella /
fibula are small and often the hardest to reconstruct), and run the paired U-Net vs V-Net Wilcoxon
test on **Dice per bone**.

In [4]:
# ---- per-bone mean +/- std (by regime x group x model x bone) + per-bone Dice Wilcoxon ----
BONES = ["femur", "tibia", "patella", "fibula"]
PB_METRICS = ["dice", "iou", "hd95_mm", "assd_mm"]

if len(df):
    d_all = df.copy(); d_all["dataset"] = "all"
    dd = pd.concat([df, d_all], ignore_index=True)
    recs = []
    for (regime, group, model), grp in dd.groupby(["regime", "dataset", "model"]):
        for bone in BONES:
            row = dict(regime=regime, group=group, model=model, bone=bone, n=len(grp))
            for m in PB_METRICS:
                col = "%s_%s" % (m, bone)
                if col in grp.columns:
                    row[m] = "%.3f +/- %.3f" % (grp[col].mean(), grp[col].std() if len(grp) > 1 else 0.0)
            recs.append(row)
    order = {"all": 0, "healthy": 1, "fractured": 2}
    pb = pd.DataFrame(recs)
    pb["__o"] = pb["group"].map(order)
    pb = pb.sort_values(["regime", "__o", "model", "bone"]).drop(columns="__o")
    print("Per-bone mean +/- std over cross-validated knees:\n")
    print(pb.to_string(index=False))
    pb.to_csv(DEC / "comparison_per_bone.csv", index=False)
    print("\nwrote ->", DEC / "comparison_per_bone.csv")

    # paired Wilcoxon on Dice per bone (reuses wilcoxon / paired_vectors / holm_bonferroni from Section 2)
    rows_pb = []
    for regime in sorted(df.regime.unique()):
        dr = df[df.regime == regime]
        for group in ["all", "healthy", "fractured"]:
            dg = dr if group == "all" else dr[dr.dataset == group]
            praw = []
            for bone in BONES:
                u, v = paired_vectors(dg, "dice_%s" % bone)
                rec = dict(regime=regime, group=group, bone=bone,
                           n_pairs=(0 if u is None else len(u)),
                           unet_dice=(np.nan if u is None else round(float(np.mean(u)), 3)),
                           vnet_dice=(np.nan if v is None else round(float(np.mean(v)), 3)),
                           p_raw=np.nan)
                if u is not None and len(u) >= 1 and np.any(u - v != 0):
                    try:
                        s, p = wilcoxon(u, v); rec["p_raw"] = float(p)
                    except ValueError:
                        pass
                praw.append(rec["p_raw"]); rows_pb.append(rec)
            adj = holm_bonferroni(praw)
            for k in range(len(BONES)):
                rows_pb[-len(BONES) + k]["p_holm"] = (None if np.isnan(adj[k]) else round(float(adj[k]), 4))
    respb = pd.DataFrame(rows_pb)
    respb["p_raw"] = respb["p_raw"].round(4)
    print("\nPer-bone Dice paired Wilcoxon (U-Net vs V-Net), Holm-corrected per regime x group:\n")
    print(respb[["regime", "group", "bone", "n_pairs", "unet_dice", "vnet_dice", "p_raw", "p_holm"]].to_string(index=False))
    respb.to_csv(DEC / "comparison_per_bone_wilcoxon.csv", index=False)
    print("\nwrote ->", DEC / "comparison_per_bone_wilcoxon.csv")
else:
    print("no data - run the decoder grid first.")

no data - run the decoder grid first.


## 4. How to read this for the thesis

- Report the **fractured** rows of the §1 table (Dice, IoU, **ASSD mm, HD95 mm**) as the primary
  fracture-reconstruction result, with the §2 Wilcoxon `p_holm` next to each — *exactly* the
  reporting style of FracReconNet ([PMC9829664](https://pmc.ncbi.nlm.nih.gov/articles/PMC9829664/))
  and the surface-supervision paper ([arXiv:2405.01204](https://arxiv.org/html/2405.01204v1)).
- Compare the **frozen** vs **finetuned** regimes: if fractured scores stay low in *both* and the
  U-Net/V-Net difference is `(ns)`, the conclusion is that **the decoder block is not the bottleneck
  for fractures** — consistent with the documented `LIFT_DEPTH=16` z-bottleneck and the
  binary-occupancy target that cannot encode a non-displaced fracture. That is the evidence that
  *triggers* the conditional fracture-aware stretch goal (auxiliary fracture class, z-bottleneck fix).
- State precisely that this is a **decoder conv-block ablation** (plain vs residual block in a shared
  ConvNeXtV2 encoder–decoder), **not** canonical 3D U-Net vs V-Net.

## 5. Gate G4 - front-end arm matrix (C-6)

Arbitrates the front-end changes (C-1 TSDF target, C-2 aux 2.5D heads, C-3 unfreeze) **one axis at a
time**, using the neutral-head Dice + surface distance (HD95/ASSD mm) that `02_frontend_pretrain.ipynb`
records per arm into `models/decoders/gate_g4_arm_metrics.csv`. Each arm ships only if it **beats the
previous accepted arm on the IRREGULAR (fractured) cohort without degrading the standard (healthy)
cohort**. Negative results are kept and printed - a rejected arm is itself a documented finding.

Run `02` once per arm (set `STEP2_TARGET`, and later the C-2/C-3 flags) across folds, then this cell.

In [5]:
# ---- Gate G4: pool arm metrics, tabulate arm x cohort, and apply the ship/no-ship rule ----
GATE_CSV = DEC / "gate_g4_arm_metrics.csv"
if not GATE_CSV.exists():
    print("[Gate G4] no arm metrics yet - run 02_frontend_pretrain.ipynb for each arm "
          "(STEP2_TARGET=binary, then tsdf_blend, ...) across folds first.")
else:
    g = pd.read_csv(GATE_CSV).drop_duplicates(subset=["arm", "dataset", "case", "side"])
    d_all = g.copy(); d_all["dataset"] = "all"
    gg = pd.concat([g, d_all], ignore_index=True)
    at = gg.groupby(["arm", "dataset"]).agg(
        n=("dice", "size"), dice=("dice", "mean"),
        hd95_mm=("hd95_mm", "mean"), assd_mm=("assd_mm", "mean")).round(3)
    print("Gate G4 - arm x cohort (mean over cross-validated knees):\n")
    print(at.to_string()); at.to_csv(DEC / "gate_g4_summary.csv")
    print("\nwrote ->", DEC / "gate_g4_summary.csv")

    # intended sequence (C-6): baseline -> +C-1 -> +C-2 -> +C-3. ref arms (imagenet/random) excluded.
    ARM_ORDER = ["binary|frozen|noaux", "tsdf|frozen|noaux", "tsdf|frozen|aux", "tsdf|unfrozen|aux"]
    present = [a for a in ARM_ORDER if a in set(g.arm.unique())]
    def row(a, ds): return at.loc[(a, ds)] if (a, ds) in at.index else None

    print("\nShip/no-ship (fractured decisive; higher Dice / lower HD95 / lower ASSD = better):")
    if not present:
        print("  (no recognized arms in the CSV yet)")
    accepted = present[0] if present else None
    if accepted: print("  baseline accepted arm:", accepted)
    for a in present[1:]:
        fa, fb = row(a, "fractured"), row(accepted, "fractured")
        ha, hb = row(a, "healthy"),   row(accepted, "healthy")
        if fa is None or fb is None:
            print("  %-22s : insufficient fractured data" % a); continue
        irr = ((fa.dice > fb.dice) + (fa.hd95_mm < fb.hd95_mm) + (fa.assd_mm < fb.assd_mm)) >= 2
        std_ok = (ha is None or hb is None) or ((ha.dice >= hb.dice - 0.01) and (ha.assd_mm <= hb.assd_mm * 1.05))
        ship = bool(irr and std_ok); verdict = "SHIP" if ship else "hold"
        print("  %-22s : %-4s | fractured dice %.3f->%.3f  assd %.3f->%.3f  hd95 %.3f->%.3f | std_ok=%s"
              % (a, verdict, fb.dice, fa.dice, fb.assd_mm, fa.assd_mm, fb.hd95_mm, fa.hd95_mm, std_ok))
        if ship: accepted = a
    print("\naccepted front-end arm after gate:", accepted)
    print("(thresholds - >=2/3 fractured metrics improve; healthy Dice not down >0.01, healthy ASSD "
          "not up >5%% - are tunable; the rule keeps the irregular cohort decisive.)")

Gate G4 - arm x cohort (mean over cross-validated knees):

                               n   dice  hd95_mm  assd_mm
arm                 dataset                              
binary|frozen|noaux all        9  0.012  162.156  112.200
                    fractured  3  0.005  171.275  118.993
                    healthy    6  0.015  157.596  108.804
tsdf|frozen|noaux   all        9  0.010  149.222   98.206
                    fractured  3  0.009  148.458   98.752
                    healthy    6  0.011  149.604   97.933

wrote -> C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\models\decoders\gate_g4_summary.csv

Ship/no-ship (fractured decisive; higher Dice / lower HD95 / lower ASSD = better):
  baseline accepted arm: binary|frozen|noaux
  tsdf|frozen|noaux      : hold | fractured dice 0.005->0.009  assd 118.993->98.752  hd95 171.275->148.458 | std_ok=True

accepted front-end arm after gate: binary|frozen|noaux
(thresholds - >=2/3 fractured metrics improve; 